# Quantifying isoform usage with Torino

Learns coverage components of each gene from RNA-seq read depth across samples with Torino, without transcript annotations, and writes their per-sample usage as a phenotype table for QTL mapping.

## Overview

This module turns aligned RNA-seq into isoform-level phenotypes without relying on a transcript annotation. Junction-based quantification such as `splicing_calling_leafcutter2` sees only the splice junctions a read crosses, so it misses intron retention and alternative polyadenylation, which change how much of a region is covered rather than which junctions are used. Annotation-based quantification sees only the isoforms the reference already lists, which under-represents those expressed in disease or in less studied tissues.

Torino models the read depth along a gene directly. For one gene, the depth of $N$ samples over $M$ bases forms a count matrix $X$, which Torino factorizes by empirical Bayes Poisson non-negative matrix factorization into $K$ coverage components and their usage in each sample, $X_{nm} \sim \text{Poisson}\big(\sum_k L_{nk} F_{mk}\big)$. Constitutive exons and introns vary little between samples, so the components that the factorization needs to explain the variation capture alternative RNA processing: cassette exons, retained introns, alternative first and last exons, and alternative polyadenylation sites. Priors on $F$ encourage biologically plausible profiles of exonic and intronic coverage while allowing for technical noise, so many components resemble isoform structures ([Buen Abad Najar et al., 2026](https://doi.org/10.64898/2026.10.21.760529)). The usage $L_{nk}$ of each component is the molecular phenotype carried forward.

**Matrix dimensions:**

- $X$: $N \times M$, read depth of each sample at each base of the gene window
- $F$: $M \times K$, the coverage components, written `EF_smooth` in the factorization output
- $L$: $N \times K$, the usage of each component in each sample, written `EL`

The module has one workflow, `torino`, whose four steps chain: read coverage from each BAM, a depth matrix per gene, a factorization per gene, and the cohort usage tables. `torino_preprocessing` runs the first two steps only.

The Torino paper goes further and merges components that share an exon chain into coverage patterns and classifies each against the annotation; that interpretation step is not part of this module. Its Alzheimer's disease analysis tested component usage directly, as this module does.

**When to run it.** After alignment with `RNA_calling`, alongside or instead of `splicing_calling_leafcutter2`. The normalized usage table and its phenotype-group file go to `phenotype_formatting` and then to the association scan, where TensorQTL's grouped permutation tests all components of one gene together.

## Input

- `--samples` **`tests/fixtures/splicing_calling/torino/protocol_example.sample_list.txt`**
(the sample manifest, tab delimited with a header. `coord_bam_list` gives each coordinate-sorted, indexed BAM relative to `--data-dir`, and `strand` is `rf`, `fr` or `strand_missing`; the regtools codes `RF`, `FR` and `XS` are accepted for the same three, so a `splicing_calling_leafcutter2` manifest can be reused. `sample_id` names every per-sample file and every column of the output tables. Output files take their prefix from this filename with all extensions stripped, so this manifest produces `protocol_example.*`; override it with `--output-prefix`.)

```
sample_id	strand	coord_bam_list
SAMPLE_001	strand_missing	bam/SAMPLE_001.bam
SAMPLE_002	strand_missing	bam/SAMPLE_002.bam
```

The 24 BAMs it names are shipped under `tests/fixtures/splicing_calling/torino/bam/`. They hold single-end reads over AP000547.3 on chr22, simulated from three of its Ensembl transcripts (fully spliced, retaining the last intron, and ending in an alternative last exon) in proportions that differ between two groups of samples. `build_fixture.R` in the same directory regenerates them and writes the simulated proportions to `simulated_isoform_proportions.tsv`.

- `--genes` **`tests/fixtures/splicing_calling/torino/protocol_example.genes.bed`**
(the genes to model, one per row: chromosome, 0-based start, end, gene ID, gene name and strand. A `#` header line is skipped. The gene ID names the per-gene files and the output phenotypes. For unstranded data, genes that overlap a gene on the opposite strand mix the two genes' reads; the Torino paper removes such regions for GTEx.)

```
#chr	start	end	gene_id	gene_name	strand
chr22	16592894	16600110	ENSG00000287285	AP000547.4	+
chr22	16601910	16615111	ENSG00000283633	AP000547.3	+
```

AP000547.4 carries no reads in the fixture, so it shows the coverage filter at work.

- `--K 3` (number of coverage components per gene. The Torino paper used 3 for its brain cohort, three regions of one tissue, and 10 across 18 GTEx tissues, where tissue differences dominate the variation.)
- `--min-counts 100` (summed depth over a gene, the sum over bases of per-base depth, below which a sample is dropped. Used by both the depth-matrix and factorization steps.)
- `--min-samples 10` (samples that must reach `--min-counts` for a gene to be modelled. The Torino paper required 100 of 2,016 GTEx samples.)
- `--flank 50` (bases added to each side of the gene.)
- `--gene-chunks 1` (number of jobs the gene list is split into for the depth matrices. Pass the same value to every step.)
- `--merge-n auto` (bases summed per matrix column to bound memory; `auto` uses 1, 3, 9 or 30 by gene width.)
- `--max-missing 0.1`, `--min-usage-q90 0.1` (the components kept for the normalized table: at most this fraction of samples missing, and a 90th-percentile usage of at least this much.)
- `--min-mapq 0` (keep only reads with at least this mapping quality; 255 keeps uniquely mapped STAR reads.)

## Output

- **`output/torino/coverage/SAMPLE_001.coverage.bedgraph.gz`** (`torino_1`, `torino_preprocessing_1`)
(per-base read depth of one sample, bgzipped bedGraph with a tabix index beside it: chromosome, 0-based start, end, depth. Zero-depth runs are reported too. A stranded sample instead gets `SAMPLE_001.plus.coverage.bedgraph.gz` and `SAMPLE_001.minus.coverage.bedgraph.gz`.)

```
chr22	0	16601911	0
chr22	16601911	16601929	1
chr22	16601929	16601948	2
```

- **`output/torino/count_matrix/ENSG00000283633.csv.gz`** (`torino_2`, `torino_preprocessing_2`)
(the depth matrix of one gene, the input Torino factorizes: one row per sample in manifest order, one column per base of the gene window named `<chrom>:<position>` with 1-based positions. Shown here are the first five bases and three bases where the first reads start.)

```
Sample_ID,chr22:16601861,chr22:16601862,chr22:16601863,chr22:16601864,chr22:16601865,...,chr22:16601919,chr22:16601920,chr22:16601921
SAMPLE_001,0,0,0,0,0,...,1,1,1
SAMPLE_002,0,0,0,0,0,...,1,1,1
```

- **`output/torino/count_matrix/protocol_example.chunk_1.list`** (`torino_2`, `torino_preprocessing_2`)
(the depth matrices this chunk wrote, one path per line; `torino_3` reads its gene list from these files)

```
output/torino/count_matrix/ENSG00000283633.csv.gz
```

- **`output/torino/count_matrix/protocol_example.chunk_1.skipped.tsv`** (`torino_2`, `torino_preprocessing_2`)
(the genes of this chunk that were not written, and why)

```
gene_id	gene_name	reason
ENSG00000287285	AP000547.4	0 sample(s) with depth >= 100, 10 needed
```

- **`output/torino/factorization/ENSG00000283633.rds`** (`torino_3`)
(the factorization of one gene, as `torino::run_torino_gene` writes it. `$torino_K3` is named for `--K` and is `NULL` if the factorization failed. `EF_smooth` holds the coverage components, bases by components, and `EL` their usage, samples by components; both carry the two Poisson-noise padding columns Torino adds at the window edges, hence 13,303 bases against the 13,301 of the depth matrix. `$coords_all` is every base column before filtering and merging.)

```
List of 3
 $ gene      : chr "ENSG00000283633"
 $ torino_K3 :List of 6
  ..$ EF       : num [1:13303, 1:3] 1.77e-10 1.77e-10 ...
  ..$ EF_smooth: num [1:13303, 1:3] 1.76e-10 1.76e-10 ...
  ..$ EL       : num [1:24, 1:3] 0.856 0.31 ...
  ..$ elbo     : num 5.86e+11
  ..$ coords   : chr [1:13303] "rpois(nsamples, 0.2)" "chr22:16601861" ...
  ..$ samples  : chr [1:24] "SAMPLE_001" "SAMPLE_002" ...
 $ coords_all: chr [1:13301] "chr22:16601861" "chr22:16601862" ...
```

On the fixture each component follows one simulated transcript across the 24 samples: the usage of `factor_1` correlates with the fully spliced transcript at 0.98, `factor_2` with the alternative last exon at 0.99, and `factor_3` with intron retention at 0.99.

- **`output/torino/protocol_example.torino_usage.bed.gz`** (`torino_4`)
(the usage of every component in every sample, one row per component keyed `<gene_id>:factor_<k>` at its gene's coordinates, coordinate-sorted, bgzipped and tabix-indexed, with `NA` for samples Torino dropped below `--min-counts`. Values rounded here.)

```
#chr	start	end	ID	SAMPLE_001	SAMPLE_002
chr22	16601910	16615111	ENSG00000283633:factor_1	0.8555	0.3102
chr22	16601910	16615111	ENSG00000283633:factor_2	0.0626	0.2320
```

- **`output/torino/protocol_example.torino_usage.qqnorm.bed.gz`** (`torino_4`)
(the phenotype for QTL mapping: the components passing `--max-missing` and `--min-usage-q90`, missing values filled with the component mean, each component centred and scaled across samples and each sample then quantile-normalized across components, as `QQNorm_factors.py` does in the Torino paper's pipeline. With one gene, each sample is normalized across three values only, hence the repeated magnitudes; a cohort run normalizes across every component of every gene.)

```
#chr	start	end	ID	SAMPLE_001	SAMPLE_002
chr22	16601910	16615111	ENSG00000283633:factor_1	0.8694	-0.8694
chr22	16601910	16615111	ENSG00000283633:factor_2	-0.8694	0.0000
```

- **`output/torino/protocol_example.torino_usage.phenotype_group.txt`** (`torino_4`)
(component ID and gene ID for every row of the normalized table, tab delimited without a header: the grouping TensorQTL's `--phenotype-group` reads)

```
ENSG00000283633:factor_1	ENSG00000283633
ENSG00000283633:factor_2	ENSG00000283633
ENSG00000283633:factor_3	ENSG00000283633
```

## Minimal Working Example

`torino` is the entry point and runs all four steps; `torino_preprocessing` stops after the per-gene depth matrices.

### Coverage components and their usage from BAMs

`torino` reads coverage from the 24 fixture BAMs, builds the depth matrix of each gene that passes the coverage filter, factorizes it into three components and writes the cohort usage tables. The factorization step needs the [torino](https://github.com/cfbuenabadn/torino) R package. The 24 fixture samples meet the default `--min-samples 10`; a cohort far larger than this one should raise it.

**Timing**: ~1 min (on toy dataset)

In [ ]:
sos run pipeline/isoform_calling_torino.ipynb torino \
    --cwd output/torino \
    --samples tests/fixtures/splicing_calling/torino/protocol_example.sample_list.txt \
    --genes tests/fixtures/splicing_calling/torino/protocol_example.genes.bed \
    --K 3

### Depth matrices only

`torino_preprocessing` stops once the per-gene depth matrices are written, for cohorts whose factorization runs elsewhere or to check the gene filter before committing to the factorization jobs.

**Timing**: ~7 s (on toy dataset)

In [ ]:
sos run pipeline/isoform_calling_torino.ipynb torino_preprocessing \
    --cwd output/torino \
    --samples tests/fixtures/splicing_calling/torino/protocol_example.sample_list.txt \
    --genes tests/fixtures/splicing_calling/torino/protocol_example.genes.bed

## Command Interface

In [ ]:
sos run pipeline/isoform_calling_torino.ipynb -h

```
usage: sos run pipeline/isoform_calling_torino.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  torino
  torino_preprocessing

Global Workflow Options:
  --modular-script-dir code/script (as path)
  --cwd output (as path)
                        The output directory for generated files.
  --samples VAL (as path, required)
                        Sample manifest: tab delimited with a header, columns
                        sample_id, strand and coord_bam_list
  --data-dir  path(f"{samples:d}")

                        Directory the coord_bam_list paths are relative to,
                        default to the same directory as the manifest
  --output-prefix  f'{samples:bnn}'

                        Prefix for every cohort-level output file; defaults to
                        the manifest name with all extensions stripped
  --job-size 1 (as int)
                        For cluster jobs, number commands to run per job
  --walltime 5h
                        Wall clock time expected
  --mem 16G
                        Memory expected
  --numThreads 1 (as int)
                        Number of threads
  --container ''
                        Software container option

Sections
  torino_1, torino_preprocessing_1:
    Workflow Options:
      --min-mapq 0 (as int)
                        keep only reads with at least this mapping quality
                        (default 0, off; 255 keeps uniquely mapped reads for
                        STAR alignments)
  torino_2, torino_preprocessing_2:
    Workflow Options:
      --genes VAL (as path, required)
                        gene BED: chrom, 0-based start, end, gene_id, gene_name,
                        strand (a #-prefixed header is skipped)
      --flank 50 (as int)
                        bases added on each side of the gene (default 50, the
                        Torino paper's value for human genes)
      --min-counts 100 (as int)
                        summed depth over the gene a sample needs to count
                        towards --min-samples (default 100)
      --min-samples 10 (as int)
                        samples that must reach --min-counts for the gene to be
                        written (default 10)
      --gene-chunks 1 (as int)
                        number of jobs the gene list is split into (default 1)
  torino_3:
    Workflow Options:
      --K 3 (as int)
                        number of coverage components per gene (default 3, the
                        Torino paper's value for its three-region brain cohort;
                        it used 10 across 18 GTEx tissues)
      --min-counts 100 (as int)
                        summed depth below which a sample is dropped before
                        factorization (default 100)
      --merge-n auto
                        bases summed per matrix column, or auto (default) to
                        choose from the gene's width
      --seed 1 (as int)
                        random seed for the factorization (default 1)
      --gene-chunks 1 (as int)
                        must match the value given to torino_2
  torino_4:
    Workflow Options:
      --genes VAL (as path, required)
                        gene BED given to torino_2
      --max-missing 0.1 (as float)
                        largest fraction of samples a kept component may be
                        missing (default 0.1)
      --min-usage-q90 0.1 (as float)
                        smallest 90th-percentile usage a kept component may have
                        (default 0.1)
```

## Workflow implementation

In [ ]:
[global]
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
# The output directory for generated files.
parameter: cwd = path("output")
# Sample manifest: tab delimited with a header, columns sample_id, strand and coord_bam_list
parameter: samples = path
# Directory the coord_bam_list paths are relative to, default to the same directory as the manifest
parameter: data_dir = path(f"{samples:d}")
# Prefix for every cohort-level output file; defaults to the manifest name with all extensions stripped
parameter: output_prefix = f'{samples:bnn}'
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 1
# Software container option
parameter: container = ""
cwd = path(f'{cwd:a}')

# Build the per-sample lists from the manifest (basic SoS input plumbing; no pandas so it runs
# in the base env). Strandness is rf, fr or strand_missing; the regtools codes RF, FR and XS
# that other splicing manifests carry are accepted for the same three. A blank or NA BAM path
# drops that sample.
_lines = open(samples).read().splitlines()
_hdr = _lines[0].split('\t')
_rows = [dict(zip(_hdr, _ln.split('\t'))) for _ln in _lines[1:] if _ln.strip()]
_alias = {'XS': 'strand_missing', 'RF': 'rf', 'FR': 'fr'}
_rows = [_r for _r in _rows if (_r.get('coord_bam_list') or 'NA').strip() not in ('', 'NA')]
sample_id  = [(_r.get('sample_id') or '').strip() for _r in _rows]
strandness = [_alias.get(_s, _s) for _s in [(_r.get('strand') or '').strip() for _r in _rows]]
bam_data   = [f'{data_dir}/{_r["coord_bam_list"].strip()}' for _r in _rows]
_bad = sorted(set(strandness) - {'strand_missing', 'rf', 'fr'})
if _bad:
    raise ValueError(f"unknown strand value(s) {_bad} in {samples}; expected rf, fr or strand_missing")
if '' in sample_id or len(set(sample_id)) != len(sample_id):
    raise ValueError(f"every row of {samples} needs a unique sample_id: coverage files are named by it")
del _lines, _hdr, _rows, _alias, _bad

### Read coverage

`torino_1` writes each BAM's per-base read depth as a bgzipped, tabix-indexed bedGraph with `torino_coverage.sh`. Spliced reads add depth to the exons they cover and none to the intron they skip. An unstranded library gives one file per sample; a stranded one gives a plus-strand and a minus-strand file, so genes that overlap on opposite strands are kept apart.

In [ ]:
[torino_1, torino_preprocessing_1]
# keep only reads with at least this mapping quality (default 0, off; 255 keeps uniquely
# mapped reads for STAR alignments)
parameter: min_mapq = 0
input: bam_data, group_by = 1, group_with = ["sample_id", "strandness"]
output: ([f'{cwd}/coverage/{_sample_id}.coverage.bedgraph.gz'] if _strandness == 'strand_missing' else
         [f'{cwd}/coverage/{_sample_id}.plus.coverage.bedgraph.gz', f'{cwd}/coverage/{_sample_id}.minus.coverage.bedgraph.gz'])
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{cwd}/coverage/{_sample_id}.stderr', stdout = f'{cwd}/coverage/{_sample_id}.stdout', container=container
    bash ${modular_script_dir}/molecular_phenotypes/calling/torino_coverage.sh \
        --bam ${_input} \
        --output-prefix ${cwd}/coverage/${_sample_id} \
        --strandness ${_strandness} \
        --min-mapq ${min_mapq}

### Per-gene depth matrices

`torino_2` cuts every gene in `--genes`, with `--flank` bases on either side, out of the coverage files with `torino_count_matrix.R`, and writes one samples-by-bases CSV per gene. A gene is written only when at least `--min-samples` samples have a summed depth of `--min-counts` over it; the rest are listed, with the reason, in the chunk's `.skipped.tsv`. `--gene-chunks` splits the gene list into that many jobs.

In [ ]:
[torino_2, torino_preprocessing_2]
# gene BED: chrom, 0-based start, end, gene_id, gene_name, strand (a #-prefixed header is skipped)
parameter: genes = path
# bases added on each side of the gene (default 50, the Torino paper's value for human genes)
parameter: flank = 50
# summed depth over the gene a sample needs to count towards --min-samples (default 100)
parameter: min_counts = 100
# samples that must reach --min-counts for the gene to be written (default 10)
parameter: min_samples = 10
# number of jobs the gene list is split into (default 1)
parameter: gene_chunks = 1
depends: genes
input: group_by = 'all', for_each = {'_chunk': list(range(1, gene_chunks + 1))}
output: f'{cwd}/count_matrix/{output_prefix}.chunk_{_chunk}.list', f'{cwd}/count_matrix/{output_prefix}.chunk_{_chunk}.skipped.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container=container
    Rscript ${modular_script_dir}/molecular_phenotypes/calling/torino_count_matrix.R \
        --samples ${samples} \
        --coverage-dir ${cwd}/coverage \
        --genes ${genes} \
        --output-dir ${cwd}/count_matrix \
        --output-list ${_output[0]} \
        --skipped ${_output[1]} \
        --flank ${flank} \
        --min-counts ${min_counts} \
        --min-samples ${min_samples} \
        --chunk ${_chunk} \
        --n-chunks ${gene_chunks}

### Factorization

`torino_3` factorizes each gene's matrix into `--K` coverage components with `torino_factorize.R`, one job per gene. Samples below `--min-counts` are dropped, and `--merge-n auto` sums consecutive bases in groups of 3, 9 or 30 above 15 kb, 60 kb and 1 Mb to bound memory. The gene list is read from the chunk lists `torino_2` wrote, so the two steps must use the same `--output-prefix` and `--gene-chunks`.

In [ ]:
[torino_3]
# number of coverage components per gene (default 3, the Torino paper's value for its three-region
# brain cohort; it used 10 across 18 GTEx tissues)
parameter: K = 3
# summed depth below which a sample is dropped before factorization (default 100)
parameter: min_counts = 100
# bases summed per matrix column, or auto (default) to choose from the gene's width
parameter: merge_n = "auto"
# random seed for the factorization (default 1)
parameter: seed = 1
# must match the value given to torino_2
parameter: gene_chunks = 1
_count_matrices = [_l.strip() for _i in range(1, gene_chunks + 1)
                   for _l in open(f'{cwd}/count_matrix/{output_prefix}.chunk_{_i}.list') if _l.strip()]
input: _count_matrices, group_by = 1
output: f'{cwd}/factorization/{_input:bnn}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container=container
    Rscript ${modular_script_dir}/molecular_phenotypes/calling/torino_factorize.R \
        --counts ${_input} \
        --output ${_output} \
        --K ${K} \
        --min-counts ${min_counts} \
        --merge-n ${merge_n} \
        --seed ${seed}

### Usage tables

`torino_4` collects every gene's component usages into a cohort phenotype table with `torino_usage.R`, and writes a normalized copy restricted to components with at most `--max-missing` missing samples and a 90th-percentile usage of at least `--min-usage-q90`, together with the component-to-gene grouping.

In [ ]:
[torino_4]
# gene BED given to torino_2
parameter: genes = path
# largest fraction of samples a kept component may be missing (default 0.1)
parameter: max_missing = 0.1
# smallest 90th-percentile usage a kept component may have (default 0.1)
parameter: min_usage_q90 = 0.1
input: group_by = 'all'
output: usage = f'{cwd}/{output_prefix}.torino_usage.bed.gz',
        qqnorm = f'{cwd}/{output_prefix}.torino_usage.qqnorm.bed.gz',
        group = f'{cwd}/{output_prefix}.torino_usage.phenotype_group.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{cwd}/{output_prefix}.torino_usage.stderr', stdout = f'{cwd}/{output_prefix}.torino_usage.stdout', container=container
    Rscript ${modular_script_dir}/molecular_phenotypes/calling/torino_usage.R \
        --factorizations ${_input} \
        --genes ${genes} \
        --samples ${samples} \
        --output-prefix ${cwd}/${output_prefix} \
        --max-missing ${max_missing} \
        --min-usage-q90 ${min_usage_q90}